# Equivalenze

In [1]:
from QuantumCompiling import CircuitState, CircuitShape
import scipy.sparse as sp
import numpy as np
import itertools as it
import csv

In [2]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ],
    3 : [ ["Sc","ID","St"] ],
    4 : [ ["Sc","Sc","St","St"] ] 
}

shape = CircuitShape(n_qubits,gates,eq_dir="eq")
shape.print_steps(); 

0: ['ID', 'ID', 'ID', 'ID']
1: ['ID', 'ID', 'Sc', 'St']
2: ['ID', 'ID', 'CZc', 'CZt']
3: ['ID', 'Sc', 'St', 'ID']
4: ['ID', 'CZc', 'CZt', 'ID']
5: ['ID', 'Sc', 'ID', 'St']
6: ['Sc', 'St', 'ID', 'ID']
7: ['Sc', 'St', 'Sc', 'St']
8: ['Sc', 'St', 'CZc', 'CZt']
9: ['CZc', 'CZt', 'ID', 'ID']
10: ['CZc', 'CZt', 'Sc', 'St']
11: ['CZc', 'CZt', 'CZc', 'CZt']
12: ['Sc', 'ID', 'St', 'ID']
13: ['Sc', 'Sc', 'St', 'St']



d = 1, ogni time-step è equivalente solo a sè stesso

In [ ]:
d=1
available_steps = shape.get_steps()

for i in range(shape.get_n_steps()**d):
    for j in range(i,shape.get_n_steps()**d):
        eq = CircuitState.equivalent_from_steps([available_steps[i]],[available_steps[j]])
        if eq: print(i,j,":",True)

d = 2. In questo caso non è necessario controllare se l'eq è composta da altre con d minore. Si implementa l'algoritmo per la determinazione delle equivalenze senza ridondanze

In [ ]:
d=2
ids = list(it.product(range(len(available_steps)), repeat=d))
eq = []
redundant_eq = []
found = []
for i in range(len(ids)):
    print(i,"/",len(ids))
    redundant_eq.append([])
    k=0
    while k < len(found):
        if found[k][1] == i:
            redundant_eq[i] = [[ids[i],eq[1]] if ids[i]!=eq[1] else [ids[i],eq[0]] for eq in redundant_eq[found[k][0]]]
            k = len(found)+1
        else:
            k += 1
    if k == len(found):
        for j in range(i+1, len(ids)):
            if CircuitState.equivalent_from_steps([available_steps[k] for k in ids[i]],[available_steps[k] for k in ids[j]]):
                eq.append([ids[i],ids[j]])
                found.append([i,j])
                redundant_eq[i].append([ids[i],ids[j]])

In [ ]:
import itertools
print("Eq:", len(eq), "; Con le ridondanti:", len(list(itertools.chain(*redundant_eq))))

Si modifica poi il metodo per scorrere su tutti i d ed eliminare le equivalenze composte da altre con d inferiore

In [ ]:
dmax = 2
redundant_eq = [ [[(i),(i)] for i in range(len(available_steps))] ]
final_eq = []
discarded = []
for d in range(2,dmax+1):
    
    ids = list(it.product(range(len(available_steps)), repeat=d))
    redundant_eq.append([])
    final_eq.append([])
    found = []
    
    for i in range(len(ids)):
        print(i,"/",len(ids))
        redundant_eq[d-1].append([])
        k=0
        while k < len(found):
            if found[k][1] == i:
                redundant_eq[d-1][i] = [[ids[i],eq[1]] if ids[i]!=eq[1] else [ids[i],eq[0]] for eq in redundant_eq[d-1][found[k][0]]]
                k = len(found)+1
            else:
                k += 1
        if k == len(found):
            for j in range(i+1, len(ids)):
                
                # check if the rule can be obtained composing shorter eq. rules
                d_i = 1
                while d_i < d:
                    if [ids[i][:d_i],ids[j][:d_i]] in redundant_eq[d_i-1] or [ids[i][d-d_i:],ids[j][d-d_i:]] in redundant_eq[d_i-1]:
                        d_i = d+1
                    else:
                        d_i += 1
                if d_i == d+1:
                    discarded.append([ids[i],ids[j]])
                if d_i == d and CircuitState.equivalent_from_steps([available_steps[k] for k in ids[i]],[available_steps[k] for k in ids[j]]):
                    final_eq[d-2].append([ids[i],ids[j]])
                    found.append([i,j])
                    redundant_eq[d-1][i].append([ids[i],ids[j]])
    print(discarded)
    redundant_eq[d-1] = list(it.chain(*redundant_eq[d-1]))

A questo punto vanno salvate su file le equivalenze effettive trovate

In [ ]:
import csv

for d in range(2,dmax+1):
    with open(f"equivalenze v2/eq{d}.csv", 'w',newline="") as fout:
        writer = csv.writer(fout, delimiter="\t")
        writer.writerow(["n_qubits:" +str(len(available_steps[0]))])
        writer.writerow(["gates:"])
        for key in gates.keys():
            writer.writerow(gates[key])
        writer.writerow(["equivalences:"])
        for eq in final_eq[d-2]:
            writer.writerow([step_id for step_id in eq[0]+eq[1]])

Controllo delle equivalenze salvate

In [8]:
d = 2
filename = f"equivalenze v2/eq{d}.csv"

with open(filename, 'r', newline='') as fin:
    reader = csv.reader(fin, delimiter="\t")
    #skips the header
    row = None
    while(row != "equivalences:"):
        row = next(reader)[0]
    
    equivalences = [[int(id) for id in row] for row in reader]

i = 8
for eq in equivalences:
    c1 = CircuitState.from_steps_ids(eq[:d],shape)
    c2 = CircuitState.from_steps_ids(eq[d:],shape)
    
    if CircuitState.equivalent(c1,c2):
        print(i,":","Equivalenza!")
        # print(c1.draw())
        # print(c2.draw())
        # print()
    else:
        raise("Non-equivalenza trovata")
    i = i+1

8 : Equivalenza!
9 : Equivalenza!
10 : Equivalenza!
11 : Equivalenza!
12 : Equivalenza!
13 : Equivalenza!
14 : Equivalenza!
15 : Equivalenza!
16 : Equivalenza!
17 : Equivalenza!
18 : Equivalenza!
19 : Equivalenza!
20 : Equivalenza!
21 : Equivalenza!
22 : Equivalenza!
23 : Equivalenza!
24 : Equivalenza!
25 : Equivalenza!
26 : Equivalenza!
27 : Equivalenza!
28 : Equivalenza!
29 : Equivalenza!
30 : Equivalenza!
31 : Equivalenza!
32 : Equivalenza!
33 : Equivalenza!
34 : Equivalenza!
35 : Equivalenza!
36 : Equivalenza!
37 : Equivalenza!
38 : Equivalenza!
39 : Equivalenza!
40 : Equivalenza!
41 : Equivalenza!
42 : Equivalenza!
43 : Equivalenza!
44 : Equivalenza!
45 : Equivalenza!
46 : Equivalenza!
47 : Equivalenza!
48 : Equivalenza!
49 : Equivalenza!
50 : Equivalenza!
51 : Equivalenza!
52 : Equivalenza!
53 : Equivalenza!
54 : Equivalenza!
55 : Equivalenza!
56 : Equivalenza!
57 : Equivalenza!
58 : Equivalenza!
59 : Equivalenza!
60 : Equivalenza!
61 : Equivalenza!
62 : Equivalenza!
63 : Equival

Si salva tutto nella classe CircuitShape

In [3]:
shape.find_equivalences(4)

0 / 196
1 / 196
2 / 196
3 / 196
4 / 196
5 / 196
6 / 196
7 / 196
8 / 196
9 / 196
10 / 196
11 / 196
12 / 196
13 / 196
14 / 196
15 / 196
16 / 196
17 / 196
18 / 196
19 / 196
20 / 196
21 / 196
22 / 196
23 / 196
24 / 196
25 / 196
26 / 196
27 / 196
28 / 196
29 / 196
30 / 196
31 / 196
32 / 196
33 / 196
34 / 196
35 / 196
36 / 196
37 / 196
38 / 196
39 / 196
40 / 196
41 / 196
42 / 196
43 / 196
44 / 196
45 / 196
46 / 196
47 / 196
48 / 196
49 / 196
50 / 196
51 / 196
52 / 196
53 / 196
54 / 196
55 / 196
56 / 196
57 / 196
58 / 196
59 / 196
60 / 196
61 / 196
62 / 196
63 / 196
64 / 196
65 / 196
66 / 196
67 / 196
68 / 196
69 / 196
70 / 196
71 / 196
72 / 196
73 / 196
74 / 196
75 / 196
76 / 196
77 / 196
78 / 196
79 / 196
80 / 196
81 / 196
82 / 196
83 / 196
84 / 196
85 / 196
86 / 196
87 / 196
88 / 196
89 / 196
90 / 196
91 / 196
92 / 196
93 / 196
94 / 196
95 / 196
96 / 196
97 / 196
98 / 196
99 / 196
100 / 196
101 / 196
102 / 196
103 / 196
104 / 196
105 / 196
106 / 196
107 / 196
108 / 196
109 / 196
110 / 196


KeyboardInterrupt: 

In [41]:
redundant_eq = [ [[(i,),(i,)] for i in range(13)] ]
print(redundant_eq)

print([(0,0)[:1],(0,0)[:1]] in redundant_eq[0])
print([(0,0)[:1],(0,0)[:1]])

[[[(0,), (0,)], [(1,), (1,)], [(2,), (2,)], [(3,), (3,)], [(4,), (4,)], [(5,), (5,)], [(6,), (6,)], [(7,), (7,)], [(8,), (8,)], [(9,), (9,)], [(10,), (10,)], [(11,), (11,)], [(12,), (12,)]]]
True
[(0,), (0,)]
